# FraudShield AI — Notebook 01: Exploratory Data Analysis

This notebook explores the PaySim synthetic financial transaction dataset to understand:
- Dataset structure and data types
- Missing values and duplicates
- Class imbalance between fraud and legitimate transactions
- Distributions of key features
- Correlations

**Dataset:** PaySim — Synthetic Financial Dataset  
**Expected location:** `../data/raw/transactions.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set consistent style
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully.')

In [ ]:
# ── Dataset Loading ───────────────────────────────────────────────
DATA_PATH = '../data/raw/transactions.csv'

if not os.path.exists(DATA_PATH):
    print('=' * 60)
    print('ERROR: Dataset not found!')
    print('=' * 60)
    print(f'Expected location: {os.path.abspath(DATA_PATH)}')
    print()
    print('SETUP INSTRUCTIONS:')
    print('1. Download the PaySim dataset from Kaggle:')
    print('   https://www.kaggle.com/datasets/mtalaltariq/paysim-data')
    print('2. Extract and rename the CSV file to: transactions.csv')
    print('3. Place it in the data/raw/ directory of this project')
    print()
    print('DEMO MODE: Loading a small synthetic sample for demonstration...')
    # Generate synthetic demo data
    np.random.seed(42)
    n = 10000
    types = np.random.choice(['CASH_IN','CASH_OUT','DEBIT','PAYMENT','TRANSFER'], n, p=[0.2,0.25,0.1,0.3,0.15])
    amounts = np.random.exponential(scale=50000, size=n).clip(1, 1e7)
    fraud_mask = np.zeros(n, dtype=int)
    # Fraudulent patterns: large TRANSFER or CASH_OUT with specific balance characteristics
    for i in range(n):
        if types[i] in ('TRANSFER','CASH_OUT') and amounts[i] > 200000:
            fraud_mask[i] = np.random.choice([0,1], p=[0.85, 0.15])
    df = pd.DataFrame({
        'step': np.random.randint(1, 744, n),
        'type': types,
        'amount': amounts.round(2),
        'nameOrig': ['C' + str(np.random.randint(1e8, 1e9)) for _ in range(n)],
        'oldbalanceOrg': np.random.exponential(scale=100000, size=n).clip(0).round(2),
        'newbalanceOrig': np.random.exponential(scale=100000, size=n).clip(0).round(2),
        'nameDest': ['C' + str(np.random.randint(1e8, 1e9)) for _ in range(n)],
        'oldbalanceDest': np.random.exponential(scale=50000, size=n).clip(0).round(2),
        'newbalanceDest': np.random.exponential(scale=50000, size=n).clip(0).round(2),
        'isFraud': fraud_mask,
        'isFlaggedFraud': 0
    })
    df.to_csv(DATA_PATH, index=False)
    print(f'Demo dataset saved to {DATA_PATH} ({len(df):,} rows, {df.isFraud.sum()} fraud)')
else:
    df = pd.read_csv(DATA_PATH)
    print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# ── Basic Info ────────────────────────────────────────────────────
print('COLUMN TYPES:')
print(df.dtypes)
print()
df.head(5)

In [ ]:
# ── Descriptive Statistics ────────────────────────────────────────
print('NUMERIC STATISTICS:')
df.describe().round(2)

In [ ]:
# ── Missing Value Analysis ────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('MISSING VALUES:')
print(missing_df[missing_df['Missing Count'] > 0])
print(f'\nTotal missing cells: {missing.sum()}')

In [ ]:
# ── Duplicate Analysis ────────────────────────────────────────────
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count:,} ({dup_count/len(df)*100:.2f}%)')

In [ ]:
# ── Class Distribution ────────────────────────────────────────────
fraud_counts = df['isFraud'].value_counts()
print('CLASS DISTRIBUTION:')
print(f'  Legitimate : {fraud_counts[0]:>10,} ({fraud_counts[0]/len(df)*100:.2f}%)')
print(f'  Fraudulent : {fraud_counts[1]:>10,} ({fraud_counts[1]/len(df)*100:.2f}%)')
print(f'  Imbalance Ratio: {fraud_counts[0]/fraud_counts[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Bar chart
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Legitimate', 'Fraudulent'], fraud_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Class Distribution (absolute)', fontweight='bold')
axes[0].set_ylabel('Transaction Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate','Fraudulent'],
            colors=colors, autopct='%1.2f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Class Distribution (percentage)', fontweight='bold')
plt.suptitle('Fraud vs Legitimate Transactions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Transaction Type Analysis ─────────────────────────────────────
type_fraud = df.groupby(['type','isFraud']).size().unstack(fill_value=0)
type_fraud.columns = ['Legitimate','Fraudulent']
type_fraud['Fraud Rate %'] = (type_fraud['Fraudulent'] / type_fraud.sum(axis=1) * 100).round(2)
print('FRAUD BY TRANSACTION TYPE:')
print(type_fraud)

ax = type_fraud[['Legitimate','Fraudulent']].plot(
    kind='bar', stacked=False, color=['#2ecc71','#e74c3c'],
    figsize=(12,5), edgecolor='white'
)
ax.set_title('Legitimate vs Fraudulent Transactions by Type', fontweight='bold', fontsize=14)
ax.set_xlabel('Transaction Type')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../data/processed/eda_type_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Amount Distribution ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df[df['isFraud']==0]['amount'].clip(upper=1e6).hist(
    bins=60, ax=axes[0], color='#2ecc71', edgecolor='white', alpha=0.8
)
axes[0].set_title('Amount Distribution — Legitimate', fontweight='bold')
axes[0].set_xlabel('Amount (capped at 1M)')
df[df['isFraud']==1]['amount'].clip(upper=1e6).hist(
    bins=60, ax=axes[1], color='#e74c3c', edgecolor='white', alpha=0.8
)
axes[1].set_title('Amount Distribution — Fraudulent', fontweight='bold')
axes[1].set_xlabel('Amount (capped at 1M)')
plt.tight_layout()
plt.savefig('../data/processed/eda_amount_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────────
numeric_cols = ['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','isFraud']
corr = df[numeric_cols].corr()
plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step (Time) Analysis ───────────────────────────────────────────
if 'step' in df.columns:
    step_fraud = df.groupby('step')['isFraud'].agg(['sum','count'])
    step_fraud.columns = ['fraud','total']
    step_fraud['fraud_rate'] = step_fraud['fraud'] / step_fraud['total']
    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax1.fill_between(step_fraud.index, step_fraud['total'], alpha=0.3, color='steelblue', label='Total')
    ax1.set_xlabel('Time Step (Hours)')
    ax1.set_ylabel('Total Transactions', color='steelblue')
    ax2 = ax1.twinx()
    ax2.plot(step_fraud.index, step_fraud['fraud_rate'], color='#e74c3c', linewidth=1.5, label='Fraud Rate')
    ax2.set_ylabel('Fraud Rate', color='#e74c3c')
    plt.title('Transaction Volume and Fraud Rate Over Time', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig('../data/processed/eda_time_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Key Observations Summary ───────────────────────────────────────
print('=' * 60)
print('KEY EDA OBSERVATIONS')
print('=' * 60)
print(f'1. Dataset has {len(df):,} transactions')
print(f'2. Fraud prevalence: {df["isFraud"].mean()*100:.2f}% — HIGHLY IMBALANCED')
print(f'3. Fraud ONLY occurs in TRANSFER and CASH_OUT types (based on PaySim)')
print(f'4. Fraudulent amounts tend to be larger on average')
print(f'5. We will use SMOTE + class_weight to handle imbalance')
print(f'6. We will NOT use isFlaggedFraud as a feature (target leakage risk)')
print(f'7. Chronological split will be used to prevent future leakage')